# Tutorial: The Data Science Lifecycle

> **Type:** Tutorial — follow along step by step. Your only job is to make it work.
> **Time:** ~1.5 hours
> **Prerequisites:** Module 02 (business problem framing), Module 09 (EDA)
> **What you'll build:** One small problem carried through an abbreviated version of the full lifecycle — problem statement, data understanding, a baseline model, an evaluation against a business metric, a deployment/monitoring plan, and a retrospective — all in a single notebook.

---

## What We're Building

This notebook does *not* teach you a new modelling technique. It's about the shape of a project. We'll take a small churn-prediction problem (the same scenario from Module 02) and move it through every lifecycle stage in miniature: frame it, look at the data, prepare it, train one deliberately simple baseline model, evaluate it against a business threshold (not just a model metric), write a real deployment and monitoring plan as a document (we don't actually deploy — that's Module 23), and finish with a retrospective that decides whether this project would loop back to stage 1.

By the end, you'll have felt the loop, not just read about it.

## Step 0: Setup

In [ ]:
# !pip install pandas numpy matplotlib scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_recall_curve, roc_auc_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['figure.figsize'] = (11, 4.5)
print('Setup complete.')

---

## Stage 1: Problem Framing

We reuse Scenario A from Module 02 ("The Telco") but restate it briefly here — in a real project this would already exist as a standalone charter document (see `howto.md`), not something you re-derive per notebook.

**Problem statement (abbreviated):**
- **Business goal:** Reduce revenue lost to preventable customer churn
- **ML problem type:** Binary classification
- **Label:** `churned_within_30d` (1 = cancelled within 30 days of observation date, 0 = did not)
- **Unit of analysis:** One row = one customer at end of billing cycle
- **Features available at prediction time:** tenure, spend, support tickets, login recency, plan type, product count, payment failures
- **Success metric (model):** Precision ≥ 0.40 at recall ≥ 0.60 — chosen because the intervention team can only act on a limited number of flagged customers, and each false positive costs a wasted outreach call
- **Success metric (business):** Beat the existing simple rule (see Stage 4) — if we can't beat a rule that already exists, this project has no reason to ship a model

> **Check:** Notice this is a *contract*, not a suggestion. Everything we do below is judged against it — not against "did the model get a good AUC."

---

## Stage 2: Data Collection & Understanding

In a real project this stage means pulling from a warehouse and running the full EDA workflow from Module 09. Here we generate a synthetic dataset that matches the schema from our problem statement, then do a fast pass of understanding — enough to catch anything that would send us back to Stage 1.

In [ ]:
N = 8_000  # 8,000 customers

months_as_customer   = np.random.exponential(scale=22, size=N).astype(int) + 1
monthly_spend        = np.random.lognormal(mean=4.4, sigma=0.6, size=N).round(2)
num_support_tickets  = np.random.poisson(lam=1.1, size=N)
days_since_login     = np.random.exponential(scale=7, size=N).astype(int)
plan_type            = np.random.choice(['basic', 'standard', 'premium'], size=N, p=[0.45, 0.4, 0.15])
num_products         = np.random.randint(1, 6, size=N)
payment_failures     = np.random.poisson(lam=0.25, size=N)

plan_risk = {'basic': 1.0, 'standard': 0.5, 'premium': 0.1}

churn_logit = (
    -3.4
    + 0.09 * num_support_tickets
    + 0.15 * payment_failures
    + 0.03 * days_since_login
    - 0.02 * np.log1p(months_as_customer)
    + np.array([plan_risk[p] for p in plan_type])
    - 0.10 * num_products
    + np.random.normal(0, 0.5, N)
)
churn_prob = 1 / (1 + np.exp(-churn_logit))
churned = (np.random.uniform(size=N) < churn_prob).astype(int)

df = pd.DataFrame({
    'customer_id':             np.arange(1, N + 1),
    'months_as_customer':      months_as_customer,
    'monthly_spend':           monthly_spend,
    'num_support_tickets_90d': num_support_tickets,
    'days_since_last_login':   days_since_login,
    'plan_type':               plan_type,
    'num_products':            num_products,
    'payment_failures_6m':     payment_failures,
    'churned_within_30d':      churned,
})

print(f"Shape: {df.shape}")
print(f"Churn rate: {df['churned_within_30d'].mean():.1%}")
df.head()

> **Data understanding checkpoint:** Churn rate should be roughly 5-10%. If it were, say, 60%, that would be a red flag worth taking back to Stage 1 — either the label window is wrong or "churn" is being defined too loosely to be actionable. This is exactly the kind of check that sends real projects backwards, not just this synthetic one.

---

## Stage 3: Data Preparation

Minimal prep here — full feature engineering is Module 14. The point at this stage of the lifecycle is just: split before you look any further, and confirm no leakage.

In [ ]:
feature_cols = [
    'months_as_customer', 'monthly_spend', 'num_support_tickets_90d',
    'days_since_last_login', 'num_products', 'payment_failures_6m',
]
label_col = 'churned_within_30d'

X = df[feature_cols]
y = df[label_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit only on train — avoids leakage
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]:,} rows | churn rate {y_train.mean():.1%}")
print(f"Test:  {X_test.shape[0]:,} rows | churn rate {y_test.mean():.1%}")

---

## Stage 4: Modelling — Baseline First

Before training anything sophisticated, we establish two baselines: a naive classifier and the "simple rule" from the problem statement. Modelling only earns its keep in Stage 5 if it beats both.

In [ ]:
# Baseline 1: naive — always predict the majority class
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_test)
print("Naive baseline (always predict 'no churn'):")
print(classification_report(y_test, dummy_preds, zero_division=0))

# Baseline 2: the simple business rule someone already proposed
# "Flag anyone with 2+ support tickets AND at least one payment failure"
rule_preds = ((X_test['num_support_tickets_90d'] >= 2) & (X_test['payment_failures_6m'] >= 1)).astype(int)
print("\nSimple rule baseline:")
print(classification_report(y_test, rule_preds, zero_division=0))

In [ ]:
# Now the actual model — deliberately simple (logistic regression).
# Module 11 onward covers model selection properly; here we only need
# something to compare against the baselines.
model = LogisticRegression(class_weight='balanced', random_state=RANDOM_SEED)
model.fit(X_train_scaled, y_train)

probs = model.predict_proba(X_test_scaled)[:, 1]
print(f"AUC-ROC: {roc_auc_score(y_test, probs):.3f}")

> **What just happened?** We didn't jump straight to "train the best model." We first proved a naive guess and a simple rule as a floor, and only then trained something more sophisticated. If the logistic regression didn't beat the rule, the correct move at this stage is to stop and go back to Stage 1 (or Stage 3) — not to keep tuning.

---

## Stage 5: Evaluation Against the Business Metric

Stage 1 said: precision ≥ 0.40 at recall ≥ 0.60. AUC alone doesn't tell us if we hit that. We need to find the threshold that gets us there, exactly like Module 02's precision/recall framing.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, probs)

# Find thresholds where recall >= 0.60, then pick the one with best precision
mask = recall[:-1] >= 0.60
if mask.any():
    best_idx = np.argmax(precision[:-1][mask])
    candidate_thresholds = thresholds[mask]
    candidate_precision = precision[:-1][mask]
    best_threshold = candidate_thresholds[best_idx]
    best_precision = candidate_precision[best_idx]
    print(f"Best threshold meeting recall >= 0.60: {best_threshold:.3f}")
    print(f"Precision at that threshold: {best_precision:.3f}")
    print(f"Target from problem statement: precision >= 0.40 -> {'MET' if best_precision >= 0.40 else 'NOT MET'}")
else:
    print("No threshold achieves recall >= 0.60 — this model does not meet the success metric.")

# Compare model's flagged count vs. the rule's flagged count, at matched recall
final_preds = (probs >= best_threshold).astype(int)
print(f"\nModel flags {final_preds.sum()} of {len(final_preds)} test customers")
print(f"Rule flags   {rule_preds.sum()} of {len(rule_preds)} test customers")
print(f"\nModel vs rule at comparable recall:")
print(classification_report(y_test, final_preds, zero_division=0))

In [ ]:
target_met = best_precision >= 0.40 if mask.any() else False
print(f"Target precision >= 0.40 at recall >= 0.60: {'MET' if target_met else 'NOT MET'}")
if not target_met:
    print("\nIn a real project, this is where you STOP and loop back -- to Stage 3 for better")
    print("features (Module 14), or to Stage 1 to renegotiate the success metric.")
    print("We deliberately kept features minimal in this tutorial so this outcome would show up --")
    print("it's the loop-back in action, not a mistake in the notebook.")

> **What just happened?** This is the moment a real project either gets a green light or gets sent back a stage. With only the raw features from Stage 3, the model likely does **not** clear the precision/recall bar agreed in Stage 1 — which is the realistic outcome of a first pass, not a bug. "The model works" is not a meaningful sentence on its own; the contract either was or wasn't met.

> **For the rest of this notebook**, we continue on to Stages 6-8 anyway — but treat that as **illustrative of what those stages look like**, not as the validated next step for *this* model. In a real project, an unmet target sends you back to Stage 3 (better features — Module 14) or Stage 1 (renegotiate the metric) *before* anyone writes a deployment plan.

---

## Stage 6: Deployment Plan (Not Deployment)

We do not deploy anything in this notebook — real deployment is Module 23 (FastAPI endpoints, containerisation, SageMaker endpoints). At this stage of the lifecycle, the deliverable is a **plan**, written down, that someone can act on.

**Deployment and monitoring plan (draft):**

- **Serving pattern:** Batch — score all active customers nightly and hand the top-N to the intervention team each morning. Real-time is not required; the intervention workflow itself takes hours, so sub-second latency has no value here (this decision comes straight from Stage 1's prediction horizon).
- **Serving owner:** ML engineering (named individual, per the charter template in `howto.md`) — not "the data science team."
- **Rollback plan:** Keep the simple rule running in parallel for the first month; if the model's flagged list looks anomalous on any given night (e.g. flags >3x the usual count), fall back to the rule automatically.
- **Retraining cadence:** Monthly, or immediately if monitoring (Stage 7) detects drift.
- **Monitoring owner:** Same ML engineer, paged via the alerting channel defined in Module 25.

> **Extension challenge:** Rewrite this plan assuming the business owner instead wants *real-time* scoring at the moment a customer contacts support. What changes about the serving pattern, latency budget, and rollback plan?

---

## Stage 7: Monitoring Plan

Also not implemented here (Module 25 covers the actual drift-detection code) — but every deployment plan needs a monitoring plan attached before it ships, not after.

**Monitoring plan (draft):**

| What we watch | Why | Alert threshold |
|---|---|---|
| Input feature distributions (e.g. mean `days_since_last_login`) | Detects data drift — the population the model sees has changed | Shift > 2 standard deviations from training distribution |
| Model's predicted churn rate | Detects model drift — if the model suddenly flags 3x more/fewer customers, something changed upstream | Deviates > 50% from the training-time flag rate |
| Realised outcome vs. prediction (30 days later) | Detects whether the model is still accurate, not just stable | Precision drops below 0.30 over a rolling 30-day window |
| Pipeline health (did the nightly job run, on the expected row count) | Catches pipeline breakage before anyone notices "no flagged customers today" | Job failure or row count outside expected range |

---

## Stage 8: Retrospective and Iteration

Every lifecycle pass ends here — deciding honestly whether the project is done, or whether it loops back.

In [ ]:
retrospective = {
    "what_we_set_out_to_do": "Build a churn model meeting precision>=0.40 at recall>=0.60 (Stage 1 contract)",
    "outcome": "MET -- proceed to real deployment" if target_met else "NOT MET -- this run loops back, not forward",
    "where_did_we_loop_back": (
        ["None needed this run -- target was met on the first pass (unusual; treat with scepticism"
         " and re-check on fresh data before celebrating)"]
        if target_met else
        ["Stage 5 -> Stage 3: raw features aren't separating churners well enough -- revisit with "
         "Module 14 feature engineering (interactions, better encodings) before re-evaluating",
         "Stage 5 -> Stage 1: if better features still don't clear the bar, the 0.40/0.60 target "
         "itself may need renegotiating with the business owner"]
    ),
    "what_we_would_do_differently": [
        "Real project: verify with the business owner that 'churn' still means the same thing it meant "
        "when the historical labels were recorded (Stage 2 risk from explanation.md)",
        "Real project: run Module 14's feature engineering before concluding the model 'is what it is' "
        "— we used raw features only, deliberately, to keep this notebook focused on the lifecycle shape",
    ],
    "next_iteration_problem_statement": (
        "If shipped: revisit in 30 days with real outcome data to check precision/recall in production "
        "match the offline evaluation. If not shipped: return to Stage 3/4 with engineered features before "
        "re-attempting evaluation."
    ),
}

for k, v in retrospective.items():
    print(f"{k}:")
    if isinstance(v, list):
        for item in v:
            print(f"  - {item}")
    else:
        print(f"  {v}")
    print()

---

## What You Built

You carried one problem through all eight lifecycle stages in a single notebook: a problem statement, an understood and prepared dataset, a baseline-beaten model, an evaluation against a *business* metric rather than a bare model metric, a written deployment plan, a written monitoring plan, and a retrospective that explicitly asks whether the project loops back to Stage 1.

## What's Next

- Read `explanation.md` if you haven't — the section on why the textbook pipeline diagram lies is the most important idea in this module.
- Open `howto.md` for the charter, pre-mortem, and retrospective templates you'll reuse on every real project.
- Keep `reference.md` open during kick-offs — the stage checklist and CRISP-DM/TDSP mapping are useful shorthand when talking to other data scientists.
- **Extension challenge:** Take the deployment plan from Stage 6 and rewrite it for a scenario where the intervention team's capacity drops from unlimited to 50 customers/night. What changes in Stage 1, Stage 5, and Stage 6?

---

**Next module:** `11_supervised_learning_overview` — now that you can see where modelling fits in the whole lifecycle, it's time to go deep on the modelling stage itself.